# Test a Multi-head YOLOv8

Back bone to 
1. Embedding head
2. Detection head

In [1]:
from ultralytics import YOLO
import torch
import torch.nn as nn
import torch.nn.parallel

In [10]:
model_path = './models/yolov8m.pt'

# Load a REGULAR model
reg_model = YOLO(model_path, 'detect')  # pretrained YOLOv8n model
reg_model.to('cuda:0')
reg_model.name = 'yolov8'

In [8]:
img_path2 = '/mnt/nas/back_ubuntu/superdave/Desktop/killboy_sm_ai/img_0405.jpg'

In [11]:
reg_model(img_path2)


image 1/1 /mnt/nas/back_ubuntu/superdave/Desktop/killboy_sm_ai/img_0405.jpg: 448x640 1 person, 2 motorcycles, 51.0ms
Speed: 2.9ms preprocess, 51.0ms inference, 3.1ms postprocess per image at shape (1, 3, 448, 640)


[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'airplane', 5: 'bus', 6: 'train', 7: 'truck', 8: 'boat', 9: 'traffic light', 10: 'fire hydrant', 11: 'stop sign', 12: 'parking meter', 13: 'bench', 14: 'bird', 15: 'cat', 16: 'dog', 17: 'horse', 18: 'sheep', 19: 'cow', 20: 'elephant', 21: 'bear', 22: 'zebra', 23: 'giraffe', 24: 'backpack', 25: 'umbrella', 26: 'handbag', 27: 'tie', 28: 'suitcase', 29: 'frisbee', 30: 'skis', 31: 'snowboard', 32: 'sports ball', 33: 'kite', 34: 'baseball bat', 35: 'baseball glove', 36: 'skateboard', 37: 'surfboard', 38: 'tennis racket', 39: 'bottle', 40: 'wine glass', 41: 'cup', 42: 'fork', 43: 'knife', 44: 'spoon', 45: 'bowl', 46: 'banana', 47: 'apple', 48: 'sandwich', 49: 'orange', 50: 'broccoli', 51: 'carrot', 52: 'hot dog', 53: 'pizza', 54: 'donut', 55: 'cake', 56: 'chair', 57: 'couch', 58: 'potted p

In [14]:
reg_model.model.model[-1]

Detect(
  (cv2): ModuleList(
    (0): Sequential(
      (0): Conv(
        (conv): Conv2d(192, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (act): SiLU(inplace=True)
      )
      (2): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1))
    )
    (1): Sequential(
      (0): Conv(
        (conv): Conv2d(384, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (act): SiLU(inplace=True)
      )
      (2): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1))
    )
    (2): Sequential(
      (0): Conv(
        (conv): Conv2d(576, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(64, 64, kernel_size=(3, 3)

In [12]:
full_model_saved = reg_model.model.model

In [14]:
reg_model.model.model = full_model_saved[:-1]

In [15]:
reg_model(img_path2)

IndexError: The shape of the mask [14, 20] at index 0 does not match the shape of the indexed tensor [576, 20, 14] at index 0

In [17]:
class MultiHeadObjectDetectionModel(nn.Module):
    def __init__(self, reg_model):
        super(MultiHeadObjectDetectionModel, self).__init__()
        
        backbone_model = reg_model
        backbone_model.model.model = backbone_model.model.model[:-1]
        
        head_model = reg_model
        head_model.model.model = head_model.model.model[-1]

        # Load a pre-trained backbone network (e.g., ResNet-50)
        self.backbone = backbone_model

        # Define the head for image embedding
        self.embedding_head = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1))
        )

        # Define the head for object detection
        self.detection_head = head_model

    def forward(self, x):
        # Backbone feature extraction
        features = self.backbone(x)

        # Image embedding
        embedding = self.embedding_head(features)

        # Detection box coordinates, probabilities, and class predictions
        detection_output = self.detection_head(features)

        return embedding, detection_output

    def _forward_heads(self, features):
        # Forward pass through the embedding head
        embedding = self.embedding_head(features)

        # Forward pass through the detection head
        detection_output = self.detection_head(features)

        return embedding, detection_output


In [19]:
reg_model(img_path2)


image 1/1 /mnt/nas/back_ubuntu/superdave/Desktop/killboy_sm_ai/img_0405.jpg: 448x640 1 person, 2 motorcycles, 15.8ms
Speed: 20.2ms preprocess, 15.8ms inference, 2.3ms postprocess per image at shape (1, 3, 448, 640)


[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'airplane', 5: 'bus', 6: 'train', 7: 'truck', 8: 'boat', 9: 'traffic light', 10: 'fire hydrant', 11: 'stop sign', 12: 'parking meter', 13: 'bench', 14: 'bird', 15: 'cat', 16: 'dog', 17: 'horse', 18: 'sheep', 19: 'cow', 20: 'elephant', 21: 'bear', 22: 'zebra', 23: 'giraffe', 24: 'backpack', 25: 'umbrella', 26: 'handbag', 27: 'tie', 28: 'suitcase', 29: 'frisbee', 30: 'skis', 31: 'snowboard', 32: 'sports ball', 33: 'kite', 34: 'baseball bat', 35: 'baseball glove', 36: 'skateboard', 37: 'surfboard', 38: 'tennis racket', 39: 'bottle', 40: 'wine glass', 41: 'cup', 42: 'fork', 43: 'knife', 44: 'spoon', 45: 'bowl', 46: 'banana', 47: 'apple', 48: 'sandwich', 49: 'orange', 50: 'broccoli', 51: 'carrot', 52: 'hot dog', 53: 'pizza', 54: 'donut', 55: 'cake', 56: 'chair', 57: 'couch', 58: 'potted p

In [20]:
# Example usage:

model = MultiHeadObjectDetectionModel(reg_model)

# Generate a sample input tensor
# input_tensor = torch.randn(1, 3, 224, 224)  # Adjust input size based on your dataset

# Forward pass to get both embedding and detection outputs
embedding, detection_output = model(img_path2)

TypeError: 'C2f' object is not iterable